In [2]:
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from torch.utils.data import random_split
import os
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset

In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [4]:
BATCH_SIZE=64

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                     std=[0.229, 0.224, 0.225])
])

In [6]:
# load the dataset
# base_train_ds = ImageFolder(root='/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color', transform=train_transform)
# base_eval_ds  = ImageFolder(root='/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color', transform=val_test_transform)
base_train_ds = ImageFolder(root='plantvill', transform=train_transform)
base_eval_ds  = ImageFolder(root='plantvill', transform=val_test_transform)

In [7]:
classes = base_train_ds.classes
class_to_idx = base_train_ds.class_to_idx

print(f"Total images found: {len(base_train_ds)}")
print(f"Total images found: {len(classes)}")
print(f"Detected Classes (Subfolders): {classes}")
print(f"Class to Index Mapping: {class_to_idx}")

Total images found: 54305
Total images found: 38
Detected Classes (Subfolders): ['Apple___Apple_scab', 'Apple___Black_rot', 'Apple___Cedar_apple_rust', 'Apple___healthy', 'Blueberry___healthy', 'Cherry_(including_sour)___Powdery_mildew', 'Cherry_(including_sour)___healthy', 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot', 'Corn_(maize)___Common_rust_', 'Corn_(maize)___Northern_Leaf_Blight', 'Corn_(maize)___healthy', 'Grape___Black_rot', 'Grape___Esca_(Black_Measles)', 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)', 'Grape___healthy', 'Orange___Haunglongbing_(Citrus_greening)', 'Peach___Bacterial_spot', 'Peach___healthy', 'Pepper,_bell___Bacterial_spot', 'Pepper,_bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Raspberry___healthy', 'Soybean___healthy', 'Squash___Powdery_mildew', 'Strawberry___Leaf_scorch', 'Strawberry___healthy', 'Tomato___Bacterial_spot', 'Tomato___Early_blight', 'Tomato___Late_blight', 'Tomato___Leaf_Mold', 'Tomato___Septoria_

In [8]:
import json
with open('class_names.json', 'w') as f:
    json.dump(classes, f)
print('Saved', len(classes), 'classes to class_names.json')


Saved 38 classes to class_names.json


In [7]:
indices = list(range(len(base_train_ds)))
train_idx, temp_idx = train_test_split(indices, test_size=0.2, random_state=42)
val_idx, test_idx   = train_test_split(temp_idx, test_size=0.5, random_state=42)

train_dataset = Subset(base_train_ds, train_idx)
val_dataset   = Subset(base_eval_ds,  val_idx)
test_dataset  = Subset(base_eval_ds,  test_idx)

# --- Fast loaders ---
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=64, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=64, shuffle=False, num_workers=4, pin_memory=True)

In [8]:
class PlantVillageCNN(nn.Module):
    def __init__(self, n_classes=38): # Changed default to 38
        super(PlantVillageCNN, self).__init__()
        
        # 1. Convolutional Layers
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=(3, 3), padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=(3, 3), padding=1)
        self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), padding=1)
        self.conv4 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), padding=1)
        self.conv5 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=(3, 3), padding=1)
        
        # Pooling Layer (Halves the height and width at each step)
        self.pool = nn.MaxPool2d(kernel_size=(2, 2))
        
        # 2. Linear/Dense Layers
        # Input size (224x224) downsampled 5 times: 224 -> 112 -> 56 -> 28 -> 14 -> 7
        # Final spatial size is 7x7. Channel depth is 64.
        self.fc1 = nn.Linear(in_features=64 * 7 * 7, out_features=512)
        self.fc2 = nn.Linear(in_features=512, out_features=n_classes) # n_classes = 38

    def forward(self, x):
        # Input: [Batch_Size, 3, 224, 224]
        x = self.pool(F.relu(self.conv1(x))) # State: [Batch_Size, 32, 112, 112]
        x = self.pool(F.relu(self.conv2(x))) # State: [Batch_Size, 64, 56, 56]
        x = self.pool(F.relu(self.conv3(x))) # State: [Batch_Size, 64, 28, 28]
        x = self.pool(F.relu(self.conv4(x))) # State: [Batch_Size, 64, 14, 14]
        x = self.pool(F.relu(self.conv5(x))) # State: [Batch_Size, 64, 7, 7]
        
        # Flatten: [Batch_Size, 64, 7, 7] -> [Batch_Size, 64 * 7 * 7]
        x = torch.flatten(x, start_dim=1)
        
        # Dense Layers
        x = F.relu(self.fc1(x))
        x = self.fc2(x) # Returns raw logits for 38 classes
        
        return x

# Instantiate for PlantVillage
model = PlantVillageCNN(n_classes=38)

In [9]:
# 1. Define the Loss Function 
# nn.CrossEntropyLoss handles 'SparseCategoricalCrossentropy' from raw logits automatically
criterion = nn.CrossEntropyLoss()

# 2. Define the Optimizer
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 3. Move model to GPU if available
model = model.to(device)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

In [10]:
EPOCHS = 40  # Set a high number, early stopping will cut it short
best_val_acc = 0.0
epochs_no_improve = 0
patience = 10

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]"):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    # --- Validation ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for images, labels in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    current_acc = val_correct / val_total

    # --- Early Stopping & Checkpoint Logic ---
    if current_acc > best_val_acc:
        best_val_acc = current_acc
        torch.save(model.state_dict(), 'best_model.pth')
        print("model saved!!")
        epochs_no_improve = 0  # Reset counter
    else:
        epochs_no_improve += 1
        print(f"No improvement for {epochs_no_improve} epochs...")

    # Check if we should stop
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered after {epoch+1} epochs.")
        print(f"Best Validation Accuracy: {best_val_acc*100:.2f}%")
        break

    # --- Scheduler ---
    scheduler.step()

    print(f"\nEpoch {epoch+1}: "
          f"Train Loss={running_loss/len(train_dataset):.4f} Acc={correct/total*100:.2f}% | "
          f"Val Loss={val_loss/len(val_dataset):.4f} Acc={current_acc*100:.2f}% | "
          f"Best Val Acc={best_val_acc*100:.2f}%")

Epoch 1/40 [Val]: 100%|██████████| 85/85 [00:14<00:00,  6.01it/s]


model saved!!

Epoch 1: Train Loss=1.3366 Acc=61.38% | Val Loss=0.5901 Acc=82.03% | Best Val Acc=82.03%


Epoch 2/40 [Val]: 100%|██████████| 85/85 [00:09<00:00,  9.21it/s]


model saved!!

Epoch 2: Train Loss=0.4543 Acc=85.75% | Val Loss=0.3571 Acc=89.52% | Best Val Acc=89.52%


Epoch 3/40 [Val]: 100%|██████████| 85/85 [00:09<00:00,  9.08it/s]


...

No improvement for 8 epochs...

Epoch 30: Train Loss=0.0006 Acc=99.99% | Val Loss=0.1563 Acc=98.34% | Best Val Acc=98.82%


Epoch 31/40 [Val]: 100%|██████████| 85/85 [00:09<00:00,  8.83it/s]


No improvement for 9 epochs...

Epoch 31: Train Loss=0.0004 Acc=100.00% | Val Loss=0.1511 Acc=98.67% | Best Val Acc=98.82%


Epoch 32/40 [Val]: 100%|██████████| 85/85 [00:09<00:00,  8.92it/s]

No improvement for 10 epochs...

Early stopping triggered after 32 epochs.
Best Validation Accuracy: 98.82%


In [11]:
# 1. Load the absolute best weights discovered during training
model.load_state_dict(torch.load('best_model.pth'))
model = model.to(device)

# 2. Set model to evaluation mode
model.eval()

test_loss = 0.0
test_correct = 0
test_total = 0

print("Starting Evaluation on Test Dataset...")

# 3. Disable gradients to maximize speed and minimize memory overhead
with torch.no_grad():
    for images, labels in tqdm(test_loader, desc="Testing"):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Track metrics
        test_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()

# 4. Compute final overall metrics
final_test_loss = test_loss / len(test_dataset)
final_test_acc = (test_correct / test_total) * 100

print("\n================ TEST RESULTS ================")
print(f"Test Loss:     {final_test_loss:.4f}")
print(f"Test Accuracy: {final_test_acc:.2f}%")
print("==============================================")

Starting Evaluation on Test Dataset...


Testing: 100%|██████████| 85/85 [00:15<00:00,  5.60it/s]


================ TEST RESULTS ================
Test Loss:     0.0708
Test Accuracy: 98.32%


find acc.